# AI Tutor - your professor for this repo

There is a professor for this course, and it comes in three forms. All three
read the same operating manual, `PROFESSOR.md` at the repository root, so they
teach the same way: diagnose the misconception first, explain the idea from
the course material, give you something to practise, then verify with the
grader and record the attempt.

| Route | What it is | Setup |
| --- | --- | --- |
| A. The agent in this repo | Claude Code, Copilot, Cursor, Codex, opencode - anything that reads `AGENTS.md` | None. Already in place. |
| B. `professor.py` | A local tutor CLI, talks to any OpenAI-compatible model (OpenAI, OpenRouter, Ollama, LM Studio) | Set one environment variable |
| C. Any chat assistant | ChatGPT, Gemini, Claude web - with `PROFESSOR.md` pasted in as the system prompt | Copy and paste |

Route A is the most useful: it can read the course notebooks *and* run the
grader for you, so the professor sees the actual state of your code.

## Route A - the agent already in this repository

`AGENTS.md` (repository root) tells any coding agent how to behave here:
read `PROFESSOR.md`, treat you as a learner, never touch the course material,
always run the grader instead of guessing, and log attempts.

Most tools pick that file up automatically:

| Tool | File it reads |
| --- | --- |
| Claude Code | `CLAUDE.md` - copy `AGENTS.md` to that name |
| GitHub Copilot | `.github/copilot-instructions.md` - copy `AGENTS.md` there |
| Cursor | `.cursor/rules/professor.mdc` - copy `AGENTS.md` there |
| Codex, opencode, Aider, Continue | `AGENTS.md` - already there |

To set up the Copilot variant:

```
copy AGENTS.md .github\copilot-instructions.md
copy AGENTS.md CLAUDE.md
```

Then just ask:

> I'm on section 02 and my `for` loop prints one item too few. Here's my code ...

The agent reads the lecture notebook for section 02, reads your notebook,
runs `grade.py` on it, and teaches from what it actually finds.

## Route B - the local tutor CLI

`20-Testing-Syllabus/professor.py` sends your question, the relevant course
material and your grading record to a model, and prints the answer.

**Point it at a model.** Any OpenAI-compatible endpoint works.

*Local models (free, no key, no data leaving your machine):*

```
pip install ollama
ollama pull llama3.1
set PROFESSOR_BASE_URL=http://localhost:11434/v1
set PROFESSOR_MODEL=llama3.1
```

*Hosted models:*

```
set PROFESSOR_API_KEY=sk-...
set PROFESSOR_MODEL=gpt-4o-mini
```

*Any other provider (OpenRouter, Together, LM Studio at
`http://localhost:1234/v1`, vLLM):* set `PROFESSOR_BASE_URL` and
`PROFESSOR_MODEL` to match.

**Ask a question**

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py ask "why does range(0, 6) give me five items?"
```

**Load a section as context** so the professor quotes the actual lecture:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py ask --section 02 "explain range again, slower"
```

**Bring the grader result in** - the professor sees which checks failed:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py ask --check "T02 - Python Statements Test.ipynb" "why did Q7 fail?"
```

**A conversation instead of one-shots**

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py office-hours --section 05
```

**Grade without a model** (useful on its own):

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py check "T02 - Python Statements Test.ipynb"
```

**See exactly what would be sent**, without calling anything:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py --offline ask "what is a decorator"
```

That prints the full payload - system prompt, blueprint, your record, the
section material - so you can see what the model receives before you trust it
with a question.

**The two written reports**

```
# review one project against its rubric
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py review --project P05 --path ../my-project

# analyse your whole record: where you are, and what to work on next
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/professor.py progress
```

Both write a report into `reviews/` and add an entry to your progress journal,
so the advice is part of your recorded history rather than a chat that scrolls
away.

## Route C - any chat assistant

No tooling, no keys in this repo.

1. Open `PROFESSOR.md` and paste it into the assistant's system prompt or
   custom instructions.
2. Paste the relevant lecture notebook (Jupyter can export one to Markdown:
   `File > Download as > Markdown`) or just the cell you are stuck on.
3. Paste your code and the grader output:

   ```
   C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus\grade.py "T02 - Python Statements Test.ipynb"
   ```

The assistant will then teach from the same manual, with the same method.

**Which route to use**

| Situation | Best route |
| --- | --- |
| You are in the repo, working on a notebook | A - the agent here can run the grader for you |
| You want a private local model | B |
| You are away from the repo, on another machine | C |
| You want the professor to check your answer | B with `--check`, or A |

## How the professor knows where you are

The professor is given three things: the question, the relevant course
material, and **your grading record** - so it can tell you to move on, or to
go back two sections and fix a gap.

The live cells below read the same record the professor sees.

In [ ]:
import os
import sys

HERE = os.getcwd()
if not os.path.exists(os.path.join(HERE, "grade_records.py")):
    HERE = r"C:/Users/andre/Learning/Udemy-Complete-Python-3-Bootcamp/20-Testing-Syllabus"
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import grade_records as gr

summary = gr.summarize()
print("Course mark : %.1f / 100 (%s)" % (summary["course_mark"] * 100, summary["band"]))
print("Signed off  : %d / %d modules" % (len(summary["signed_off"]), len(summary["rows"])))
print("Next target : %s" % (summary["next_target"] or "none - take the final exam"))
print()
for row in summary["rows"]:
    test = row["test"]
    best = row["best"]
    mark = "%2d/%d" % (best, test["points_possible"]) if best is not None else "  -/%d" % test["points_possible"]
    print("%-4s %-36s %-7s %s" % (test["id"], test["title"][:36], mark,
                                   gr.format_bar(best, test["points_possible"])))

## Office hours - what to ask

| When you are | Ask this |
| --- | --- |
| Stuck on one concept | "I'm on section 02 and I don't understand why `range(0, 6)` gives five items. Start with a diagnostic question." |
| Code does not work | "Here is my code and the grader output. What is the misconception, not the fix?" |
| Want the answer | "Give me the reference solution for Q4 of T02 and explain the line that trips people up." |
| Want it checked | "Grade T02 and tell me which single idea is weakest." |
| Choosing what to learn | "Here is my record. What should I study next and why?" |
| Porting to a job | "Which parts of this course matter most for backend work, and which can wait?" |
| Want a harder exercise | "Give me a harder variant of section 05's decorator exercise." |

## How to ask a good question

The fastest answers come with four things attached:

1. **Where you are** - the section number, or the test id (`T02`).
2. **What you expected** - one sentence.
3. **What happened** - the code, and the grader output.
4. **What you already tried** - so the professor does not repeat it.

A template:

```
Section: 05 (T05)
Expected: the Account class should print the balance when I print it.
Happened: print(account) shows <__main__.Account object at 0x...>
Tried: __str__, but I put it outside the class.
Question: what is actually called when you print an object?
```

The professor can then diagnose from the course material rather than guessing.

## Rules of engagement

The professor follows a ladder, so you always get the least help that unblocks
you:

1. A pointer to the exact notebook and cell.
2. A question that exposes the misconception.
3. A worked micro-example.
4. The reference solution, line by line.
5. The finished answer - only when you ask for it, and always followed by
   "what would you notice about this?".

Retakes are unlimited and free. The professor will not let a section pass on
a score it did not earn, and it will not invent a test result, a traceback,
or content that is not in this repository.

## Safety

- Keep API keys in environment variables. Never paste one into a notebook, a
  commit, or a chat message - and never paste customer data into a prompt.
- Route B with a local model (Ollama, LM Studio) sends nothing off your
  machine. Route A and Route C send your question to whatever model that tool
  is configured to use.
- The professor may run `grade.py` on your notebooks; it never edits the
  course material in sections `00-` to `19-`.